# F1 Race Prediction – EDA & Preprocessing
Dataset: `f1_2018_2024_wow_master_dataset.csv` (seasons 2018–2024)

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

FILE_PATH  = "../data/raw/f1_2018_2024_wow_master_dataset.csv"
OUTPUT_DIR = "../reports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(FILE_PATH):
    df = pd.read_csv(FILE_PATH)
    print(f"File loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Seasons: {sorted(df['season'].unique())}")
    display(df.head())
else:
    print(f"Error: file not found at {os.path.abspath(FILE_PATH)}")

## New Engineered Feature – `driver_season_momentum`
Average championship points earned per race so far in the season **before** this event.  
Formula: `driver_season_points_before_race / driver_season_races_before_race`  
No data leakage: all values are known before the race starts.

In [ ]:
df["driver_season_momentum"] = (
    df["driver_season_points_before_race"]
    / df["driver_season_races_before_race"].replace(0, np.nan)
).fillna(0.0)

print("driver_season_momentum stats:")
display(df["driver_season_momentum"].describe().round(2))

## Insight 1 – Grid Position Effect

In [ ]:
finished = df[(df["finished_flag"] == 1) & (df["grid"] > 0)]
corr = finished["grid"].corr(finished["finish_position"])

avg_finish_by_grid = (
    finished.groupby("grid")["finish_position"]
    .mean()
    .reset_index()
    .rename(columns={"finish_position": "avg_finish"})
)
top5_avg = avg_finish_by_grid[avg_finish_by_grid["grid"] <= 5]["avg_finish"].mean()
mid_avg  = avg_finish_by_grid[
    (avg_finish_by_grid["grid"] >= 6) & (avg_finish_by_grid["grid"] <= 10)
]["avg_finish"].mean()

print(f"Pearson correlation (grid vs finish): {corr:.3f}")
print(f"Avg finish for P1-P5 starters:        {top5_avg:.2f}")
print(f"Avg finish for P6-P10 starters:       {mid_avg:.2f}")
print(f"Front-row starters finish ~{mid_avg - top5_avg:.1f} places better on average.")

## Insight 2 – Pit Stop Strategy

In [ ]:
pit_df = df[
    (df["finished_flag"] == 1)
    & (df["pit_stop_count"] > 0)
    & (df["pit_stop_count"] <= 4)
]
summary = (
    pit_df.groupby("pit_stop_count")["finish_position"]
    .agg(avg="mean", count="size")
    .reset_index()
)
display(summary)

one_stop = summary.loc[summary["pit_stop_count"] == 1, "avg"].values
two_stop = summary.loc[summary["pit_stop_count"] == 2, "avg"].values
if one_stop.size and two_stop.size:
    print(f"2-stop vs 1-stop avg finish gap: {two_stop[0] - one_stop[0]:+.2f} positions")

## Insight 3 – Constructor DNF Rates

In [ ]:
constructors = df.groupby("constructor_name").agg(
    total_starts=("dnf_flag", "size"),
    total_dnfs=("dnf_flag", "sum"),
)
constructors["dnf_rate_pct"] = (
    constructors["total_dnfs"] / constructors["total_starts"] * 100
).round(1)
constructors = constructors[constructors["total_starts"] >= 30].sort_values("dnf_rate_pct")
display(constructors[["total_starts", "total_dnfs", "dnf_rate_pct"]])

## EDA Chart – Average Finish Position by Starting Grid (P1–P20)

In [ ]:
finished_grid = df[(df["finished_flag"] == 1) & (df["grid"].between(1, 20))]
stats = (
    finished_grid.groupby("grid")["finish_position"]
    .agg(["mean", "median", "std"])
    .reset_index()
)
stats.columns = ["grid", "mean", "median", "std"]

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(
    stats["grid"],
    (stats["mean"] - stats["std"]).clip(lower=1),
    (stats["mean"] + stats["std"]).clip(upper=20),
    alpha=0.2, color="#E10600", label="±1 std dev",
)
ax.plot(stats["grid"], stats["mean"],   color="#E10600", linewidth=2.5, marker="o", label="Mean finish")
ax.plot(stats["grid"], stats["median"], color="#1f77b4", linewidth=1.5, linestyle="--", marker="s", label="Median finish")
ax.plot(stats["grid"], stats["grid"],   color="gray",    linewidth=1,   linestyle=":",  label="Grid = Finish (no change)")

ax.set_xlabel("Starting Grid Position", fontsize=13)
ax.set_ylabel("Finish Position", fontsize=13)
ax.set_title("Average Finish Position by Starting Grid (2018–2024)", fontsize=15, fontweight="bold")
ax.set_xticks(range(1, 21))
ax.set_yticks(range(1, 21))
ax.invert_yaxis()
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
sns.despine(ax=ax)
fig.tight_layout()

out_path = os.path.join(OUTPUT_DIR, "eda_grid_vs_finish.png")
fig.savefig(out_path, dpi=150)
print(f"Chart saved → {out_path}")
plt.show()

## Ceren's Linear Regression
Train on 2018–2023, test on 2024.  
Features: `grid`, `driver_form_score`, `weekend_readiness`, `driver_season_momentum`

In [ ]:
# --- feature engineering ---
frame = df.copy()
frame = frame[(frame["finish_position"].notna()) & (frame["grid"] > 0)].copy()
frame["qualifying_position"] = frame["qualifying_position"].fillna(frame["grid"])

frame["driver_form_score"] = (
    0.7 * frame["last_3_race_avg_finish"] + 0.3 * frame["last_5_race_avg_finish"]
)
frame["weekend_readiness"] = (
    0.5 * frame["qualifying_position"]
    + 0.3 * frame["driver_prev_circuit_avg_finish"]
    + 0.2 * frame["driver_championship_position_before_race"]
)

# --- train / test split ---
TARGET   = "finish_position"
FEATURES = ["grid", "driver_form_score", "weekend_readiness", "driver_season_momentum"]

train_df = frame[frame["season"] <= 2023].copy()
test_df  = frame[frame["season"] == 2024].copy()
print(f"Train rows: {len(train_df)}  |  Test rows: {len(test_df)}")

# --- fit via normal equations ---
x_train = np.column_stack([np.ones(len(train_df)), train_df[FEATURES].to_numpy(float)])
x_test  = np.column_stack([np.ones(len(test_df)),  test_df[FEATURES].to_numpy(float)])
y_train = train_df[TARGET].to_numpy(float)
y_test  = test_df[TARGET].to_numpy(float)

beta, *_ = np.linalg.lstsq(x_train, y_train, rcond=None)
y_pred   = np.clip(x_test @ beta, 1, 20)

mae_val  = float(np.mean(np.abs(y_test - y_pred)))
rmse_val = float(np.sqrt(np.mean((y_test - y_pred) ** 2)))
ss_res   = float(np.sum((y_test - y_pred) ** 2))
ss_tot   = float(np.sum((y_test - np.mean(y_test)) ** 2))
r2_val   = float(1 - ss_res / ss_tot) if ss_tot else 0.0

metrics = {
    "train_seasons": "2018-2023", "test_season": 2024,
    "rows_train": len(train_df), "rows_test": len(test_df),
    "target": TARGET, "features_used_in_model": FEATURES,
    "engineered_features": {
        "driver_form_score": "0.7 * last_3_race_avg_finish + 0.3 * last_5_race_avg_finish",
        "weekend_readiness": "0.5 * qualifying_position + 0.3 * driver_prev_circuit_avg_finish + 0.2 * driver_championship_position_before_race",
        "driver_season_momentum": "driver_season_points_before_race / driver_season_races_before_race",
    },
    "coefficients": {
        "intercept": float(beta[0]),
        "grid": float(beta[1]),
        "driver_form_score": float(beta[2]),
        "weekend_readiness": float(beta[3]),
        "driver_season_momentum": float(beta[4]),
    },
    "metrics": {"mae": mae_val, "rmse": rmse_val, "r2": r2_val},
}
print(json.dumps(metrics["metrics"], indent=2))

# --- save outputs ---
result_df = test_df[
    ["season", "race_round", "race_name", "driver_full_name", "constructor_name",
     "grid", "qualifying_position", "driver_form_score", "weekend_readiness",
     "driver_season_momentum", TARGET]
].copy()
result_df["predicted_finish_position"] = np.round(y_pred, 2)
result_df["absolute_error"] = np.round(np.abs(result_df[TARGET] - result_df["predicted_finish_position"]), 2)

pred_path    = os.path.join(OUTPUT_DIR, "ceren_linear_regression_predictions_2024.csv")
metrics_path = os.path.join(OUTPUT_DIR, "ceren_linear_regression_metrics.json")
result_df.to_csv(pred_path, index=False)
open(metrics_path, "w").write(json.dumps(metrics, indent=2))
print(f"Predictions saved → {pred_path}")
display(result_df.head(10))

## Kevser's Model Diagnostics
Three panels evaluating Ceren's regression on the 2024 test set:  
**Actual vs. Predicted** · **Feature Coefficients** · **Residuals vs. Predicted**

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1 – Actual vs. Predicted
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.35, color="#E10600", s=18)
ax.plot([1, 20], [1, 20], "k--", linewidth=1, label="Perfect prediction")
ax.set_xlim(1, 20)
ax.set_ylim(1, 20)
ax.set_xlabel("Actual Finish Position", fontsize=12)
ax.set_ylabel("Predicted Finish Position", fontsize=12)
ax.set_title("Actual vs. Predicted", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
sns.despine(ax=ax)

# 2 – Feature Coefficients (excluding intercept)
ax = axes[1]
coef_vals  = beta[1:]
colors     = ["#E10600" if v > 0 else "#1f77b4" for v in coef_vals]
bars       = ax.barh(FEATURES, coef_vals, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
for bar, val in zip(bars, coef_vals):
    ax.text(
        val + (0.01 if val >= 0 else -0.01),
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va="center",
        ha="left" if val >= 0 else "right", fontsize=9,
    )
ax.set_xlabel("Coefficient Value", fontsize=12)
ax.set_title("Feature Coefficients", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
sns.despine(ax=ax)

# 3 – Residuals vs. Predicted
ax = axes[2]
ax.scatter(y_pred, residuals, alpha=0.35, color="#1f77b4", s=18)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Predicted Finish Position", fontsize=12)
ax.set_ylabel("Residual (Actual − Predicted)", fontsize=12)
ax.set_title("Residuals vs. Predicted", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)
sns.despine(ax=ax)

fig.suptitle(
    "Kevser's Model Diagnostics – Ceren's Linear Regression (2024 Test Set)",
    fontsize=14, fontweight="bold",
)
fig.tight_layout()

diag_path = os.path.join(OUTPUT_DIR, "ceren_lr_diagnostics.png")
fig.savefig(diag_path, dpi=150)
print(f"Diagnostics chart saved → {diag_path}")
plt.show()